# 03 · SpamAssassin spam — LLM-разметка (плоский JSONL)

Источник: `data/raw/collected/spamassassin.jsonl` → подвыборка `block_ab/samples/sa_spam_stratified.jsonl`.

**Выход:** `data/interim/annotated/spamassassin_spam_annotated.jsonl` — **тот же плоский формат**, что и `sms_spam_annotated.jsonl`:
`text`, `label`, … исходные поля + `scenario_family`, `annotation_confidence`, `annotation_model` (+ опционально `annotated_at`, флаги).

**Кеш:** повторный запуск не дублирует записи (ключ `md5` от текста, как в ноутбуке 01).

**Миграция:** если остался старый вложенный `block_ab/cache/sa_spam.jsonl`, ячейка ниже переносит его в плоский файл без потери строк.


In [1]:
# CONFIG
from __future__ import annotations
RANDOM_SEED = 42
N_SA_SPAM_UNIQUE = 700
BATCH_SIZE = 12
SLEEP_SEC = 2.5
MAX_NEW_THIS_RUN = 1
ANNOTATION_MODEL = "openai/gpt-4o-mini"


In [2]:
# Setup
import importlib.util
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI, APIError, RateLimitError
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from tqdm.auto import tqdm
import pandas as pd

def _find_v2_root() -> Path:
    candidate = Path(globals().get("__vsc_ipynb_file__", globals().get("__file__", "."))).resolve()
    for p in [candidate, *candidate.parents]:
        if (p / "pyproject.toml").exists() and (p / "notebooks").exists():
            return p
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "notebooks").exists():
            return p
    raise RuntimeError("Cannot find v2/ project root")

V2_ROOT = _find_v2_root()
RAW_DIR = V2_ROOT / "data" / "raw" / "collected"
INTERIM = V2_ROOT / "data" / "interim" / "annotated"
BLOCK_AB = INTERIM / "block_ab"
SAMP_DIR = BLOCK_AB / "samples"
OUT_FLAT = INTERIM / "spamassassin_spam_annotated.jsonl"
LEGACY_NESTED = BLOCK_AB / "cache" / "sa_spam.jsonl"

for d in (BLOCK_AB, SAMP_DIR, INTERIM):
    d.mkdir(parents=True, exist_ok=True)

_common_path = V2_ROOT / "notebooks" / "02_dataset_design" / "_ann_common.py"
spec = importlib.util.spec_from_file_location("_ann_common", _common_path)
ac = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ac)

env_file = V2_ROOT / ".env"
load_dotenv(env_file if env_file.exists() else None)
api_key = os.environ.get("OPENROUTER_API_KEY", "")
if not api_key:
    raise EnvironmentError("OPENROUTER_API_KEY missing in v2/.env")
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)
print("OUT_FLAT =", OUT_FLAT)


OUT_FLAT = /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/annotated/spamassassin_spam_annotated.jsonl


In [3]:
# Optional: migrate nested block_ab/cache/sa_spam.jsonl → flat (idempotent)
if LEGACY_NESTED.exists():
    w, s = ac.migrate_nested_cache_to_flat(LEGACY_NESTED, OUT_FLAT, skip_existing=True)
    print(f"Legacy migration: wrote {w} new flat rows, skipped {s} duplicates already in OUT_FLAT")
else:
    print("No legacy nested cache at", LEGACY_NESTED)


Legacy migration: wrote 540 new flat rows, skipped 0 duplicates already in OUT_FLAT


In [ ]:
# Build / load stratified sample (frozen file)
sa_all = ac.load_jsonl(RAW_DIR / "spamassassin.jsonl")
sa_spam = [r for r in sa_all if r.get("label") == "spam"]

def build_sa_spam_sample():
    uniq = ac.dedupe_records_by_text_sha(sa_spam)
    rows = []
    for r in uniq:
        txt = r.get("text") or ""
        rows.append({
            **r,
            "_stratum_arch": r.get("archive", ""),
            "_stratum_wc": ac.wc_bin(ac.wc(txt)),
            "_stratum_url": str(bool(r.get("has_url"))),
            "_stratum_proxy": ac.proxy_bucket_spam(txt),
        })
    df = pd.DataFrame(rows)
    picked = ac.stratified_sample_df(
        df, N_SA_SPAM_UNIQUE,
        ["_stratum_arch", "_stratum_wc", "_stratum_url", "_stratum_proxy"],
        RANDOM_SEED + 7,
    )
    return picked.drop(columns=[c for c in picked.columns if c.startswith("_stratum_")]).to_dict("records")

sample_path = SAMP_DIR / "sa_spam_stratified.jsonl"
sample_sa = ac.ensure_sample(sample_path, build_sa_spam_sample)
print("Sample size:", len(sample_sa))


In [ ]:
# Prompts + API
SPAM_EMAIL_CLASSES = '''
Exactly one "category" (stored as scenario_family):
- phishing_email
- candidate_419_or_advance_fee
- promo_marketing_email
- generic_spam_nonphishing
- malware_or_attachment_lure
- unclear_other
Also booleans: has_financial_pretence, has_credentials_request, has_reward_or_prize,
has_confidentiality_appeal, core_candidate
"confidence": "high" | "medium" | "low"
'''

def wrap_spam_email(r: dict) -> list[dict]:
    subj = (r.get("subject") or "").strip()
    body = (r.get("text") or "").strip()
    user = f"Source: SpamAssassin (label=spam).\nSubject: {subj}\n\nBody:\n{body}"
    return [
        {"role": "system", "content": "You annotate email spam for dataset design. " + SPAM_EMAIL_CLASSES + "\nRespond JSON only: category, confidence, has_financial_pretence, has_credentials_request, has_reward_or_prize, has_confidentiality_appeal, core_candidate."},
        {"role": "user", "content": user},
    ]

OK_SPAM = {"phishing_email", "candidate_419_or_advance_fee", "promo_marketing_email",
           "generic_spam_nonphishing", "malware_or_attachment_lure", "unclear_other"}

def validate_spam_email_ann(d: dict) -> dict:
    cat = d.get("category", "unclear_other")
    if cat not in OK_SPAM:
        cat = "unclear_other"
    conf = d.get("confidence", "low")
    if conf not in {"high", "medium", "low"}:
        conf = "low"
    def b(x): return bool(x)
    return {
        "category": cat, "confidence": conf,
        "has_financial_pretence": b(d.get("has_financial_pretence")),
        "has_credentials_request": b(d.get("has_credentials_request")),
        "has_reward_or_prize": b(d.get("has_reward_or_prize")),
        "has_confidentiality_appeal": b(d.get("has_confidentiality_appeal")),
        "core_candidate": b(d.get("core_candidate")),
    }

@retry(retry=retry_if_exception_type((RateLimitError, APIError)), wait=wait_exponential(multiplier=2, min=2, max=45), stop=stop_after_attempt(6))
def call_json(messages: list[dict]) -> dict:
    resp = client.chat.completions.create(
        model=ANNOTATION_MODEL, messages=messages,
        response_format={"type": "json_object"}, temperature=0, max_tokens=256,
    )
    return json.loads(resp.choices[0].message.content)


In [ ]:
# Annotate (batched + pause); append flat rows only for new md5 keys
idx = ac.load_flat_annotation_index(OUT_FLAT)
pending = [r for r in sample_sa if ac.md5_text_key(r.get("text", "")) not in idx]
print(f"Already annotated: {len(idx):,} | Pending: {len(pending):,}")

def run_batch():
    n_done = 0
    batch_i = 0
    for r in tqdm(pending[:MAX_NEW_THIS_RUN], desc="sa_spam"):
        k = ac.md5_text_key(r.get("text", ""))
        if k in idx:
            continue
        try:
            raw_ann = call_json(wrap_spam_email(r))
            ann = validate_spam_email_ann(raw_ann)
        except Exception as e:
            ann = validate_spam_email_ann({"category": "unclear_other", "confidence": "low"})
            ann["_error"] = str(e)[:200]
        ts = datetime.now(timezone.utc).isoformat()
        extra = {x: ann[x] for x in ("core_candidate", "has_financial_pretence", "has_credentials_request",
               "has_reward_or_prize", "has_confidentiality_appeal") if x in ann}
        if "_error" in ann:
            extra["_error"] = ann["_error"]
        flat = ac.make_flat_record(
            r, scenario_family=ann["category"], annotation_confidence=ann["confidence"],
            annotation_model=ANNOTATION_MODEL, annotated_at=ts, extra=extra,
        )
        ac.append_jsonl(OUT_FLAT, flat)
        idx[k] = flat
        n_done += 1
        batch_i += 1
        if batch_i >= BATCH_SIZE:
            batch_i = 0
            time.sleep(SLEEP_SEC)
    print(f"New annotations this run: {n_done}")

run_batch()
print("Total in index now:", len(ac.load_flat_annotation_index(OUT_FLAT)))


In [ ]:
# Quick distribution
import pandas as pd
rows = ac.load_jsonl(OUT_FLAT)
df = pd.DataFrame([r for r in rows if r.get("scenario_family")])
print(df["scenario_family"].value_counts())
print(df["annotation_confidence"].value_counts())
